# 1. Environment Setup and Module Loading
Set up paths, import project modules, and reload local code so retrieval reflects the latest implementation.

In [1]:
import os
import sys
import importlib

# Get the absolute path of the notebook's current directory
notebook_dir = os.getcwd()

# Go up one level to get the project root directory
project_root = os.path.dirname(notebook_dir)

# Add the project root to the Python path
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import src.pagerank as pagerank_module
import src.graph_builder as graph_builder_module
import src.graph_retriever as graph_retriever_module

# Reload local modules so notebook picks up recent code edits
importlib.reload(pagerank_module)
importlib.reload(graph_builder_module)
importlib.reload(graph_retriever_module)

PageRankEngine = pagerank_module.PageRankEngine
KnowledgeGraphBuilder = graph_builder_module.KnowledgeGraphBuilder
GraphRAGRetriever = graph_retriever_module.GraphRAGRetriever

# 2. Graph Construction and Baseline Retrieval
Build the mock knowledge graph, initialize the GraphRAG retriever, and run top-k retrieval for the Marie Curie multi-hop query.

In [2]:
# 1. INGESTION: Start from fixed triplets (mock ingestion layer)
builder = KnowledgeGraphBuilder()
document_text = "Marie Curie was a physicist who discovered Radium..."
edges = builder.extract_edges_mock(document_text)

# 2. ORCHESTRATION: Initialize Retriever with PageRank + bridge-aware ranking
retriever = GraphRAGRetriever(
    edges,
    bridge_weight=0.35,
    max_hops=3,
    include_seed_nodes=False,
    seed_top_k=2,
    use_undirected_expansion=True,
 )

# 3. RETRIEVAL ONLY: Vector seeds -> expansion -> reranking -> top-k
query = "What discoveries by Marie Curie led to later advances in medical imaging?"
result = retriever.retrieve_context_with_facts(query, top_k=5)

print("Seeds:", result["seeds"])
print("Top Nodes:", result["top_nodes"])
print("Context Facts:", result["context_facts"])
print("Scores:")
for item in result["scores"]:
    print(item)

Extracting entities and relationships (Mock Mode)...
Graph updated: Currently holds 14 nodes and 14 edges.
Initializing GraphRAG Retriever...
Extracting nodes and building index mappings...
Processing 14 unique nodes and 14 edges...
Sparse transition matrix M successfully built.

Starting Iterative Solver (Power Method)...
-> Converged in 24 iterations (0.0010 seconds).
Seeds: ['Marie Curie', 'Medical Imaging']
Top Nodes: ['Radiotherapy', 'X-Rays', 'Radioactivity', 'Modern Diagnostics', 'Cancer Treatment']
Context Facts: ['Radioactivity -> X-Rays', 'X-Rays -> Radiotherapy', 'Radiotherapy -> Cancer Treatment']
Scores:
{'node': 'Radiotherapy', 'final': 0.874, 'pagerank': 1.0, 'bridge': 0.64, 'hop': 1}
{'node': 'X-Rays', 'final': 0.7727271487039055, 'pagerank': 0.8441956133906239, 'bridge': 0.64, 'hop': 2}
{'node': 'Radioactivity', 'final': 0.6535825913993107, 'pagerank': 0.660896294460478, 'bridge': 0.64, 'hop': 2}
{'node': 'Modern Diagnostics', 'final': 0.5564179256332078, 'pagerank': 0

# 3. Global vs Personalized PageRank Retrieval Comparison
Compare global PageRank ranking against seed-biased personalized PageRank to evaluate query-focused retrieval behavior.

In [3]:
# 4. PERSONALIZED PAGERANK: Compare with query-aware authority scoring
query = "What discoveries by Marie Curie led to later advances in medical imaging?"

# Global PageRank 
result_global = retriever.retrieve_context_with_facts(query, top_k=5)
print("\n[GLOBAL PAGERANK]")
print(f"Seeds: {result_global['seeds']}")
print(f"Top Nodes: {result_global['top_nodes']}")
print("Scores:")
for item in result_global['scores']:
    print(
        f"  {item['node']:25} | final={item['final']:.3f} | "
        f"pagerank={item['pagerank']:.3f} | bridge={item['bridge']:.3f}"
    )

print("=" * 80)
# Personalized PageRank variant (biased toward seeds)
result_ppr = retriever.retrieve_context_with_personalized_pagerank(query, top_k=5)
print("\n[PERSONALIZED PAGERANK (biased toward seeds)]")
print(f"Seeds: {result_ppr['seeds']}")
print(f"Top Nodes: {result_ppr['top_nodes']}")
print("Scores:")
for item in result_ppr['scores']:
    print(
        f"  {item['node']:25} | final={item['final']:.3f} | "
        f"pagerank={item['pagerank']:.3f} | bridge={item['bridge']:.3f}"
    )


[GLOBAL PAGERANK]
Seeds: ['Marie Curie', 'Medical Imaging']
Top Nodes: ['Radiotherapy', 'X-Rays', 'Radioactivity', 'Modern Diagnostics', 'Cancer Treatment']
Scores:
  Radiotherapy              | final=0.874 | pagerank=1.000 | bridge=0.640
  X-Rays                    | final=0.773 | pagerank=0.844 | bridge=0.640
  Radioactivity             | final=0.654 | pagerank=0.661 | bridge=0.640
  Modern Diagnostics        | final=0.556 | pagerank=0.764 | bridge=0.171
  Cancer Treatment          | final=0.428 | pagerank=0.566 | bridge=0.171
Starting Personalized PageRank Solver (biased toward 2 seeds)...
-> PPR Converged in 52 iterations (0.0020 seconds).

[PERSONALIZED PAGERANK (biased toward seeds)]
Seeds: ['Marie Curie', 'Medical Imaging']
Top Nodes: ['Modern Diagnostics', 'Radioactivity', 'X-Rays', 'Radiotherapy', 'Radium']
Scores:
  Modern Diagnostics        | final=0.604 | pagerank=0.837 | bridge=0.171
  Radioactivity             | final=0.354 | pagerank=0.200 | bridge=0.640
  X-Rays       